In [ ]:
import operator
from typing import TypedDict, List, Dict, Any
from typing_extensions import Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langgraph.checkpoint.memory import MemorySaver

from langchain.messages import HumanMessage, SystemMessage
from langchain_gigachat.chat_models import GigaChat
import os 
from dotenv import load_dotenv
from rich import print
import pandas as pd
from pathlib import Path
import pandas as pd
from random import randint
from pydantic import BaseModel, Field


load_dotenv()
key = os.environ.get('GIGACHAT_API_KEY')

class Answer(BaseModel):
    """A structured output model for the generated question and answer pair."""
    question: str = Field(..., description="The generated question related to the cybersecurity domain.")
    answer: str = Field(..., description="The generated answer providing accurate and informative information related to the cybersecurity domain.")

llm_quesitons = GigaChat(
    credentials=key,
    model='GigaChat-2-Max',
    scope='GIGACHAT_API_CORP',
    temperature=1,
    verify_ssl_certs=False,
    profanity_check=False,
    max_tokens=250,
    timeout=300,
)

llm_answer = GigaChat(
    credentials=key,
    model='GigaChat-2-Max',
    scope='GIGACHAT_API_CORP',
    temperature=0.3,
    verify_ssl_certs=False,
    profanity_check=False,
    max_tokens=3000,
    timeout=300,
)

domains = ["computer viruses",
                   "fraud detection",
                   "phishing attacks",
                   "malware analysis",
                   "common cyber security questions",]



qa_df = pd.DataFrame(columns=["domains","question", "answer", "is_complaint"])

for i in range(100):
    selected_domain = domains[randint(0, len(domains)-1)]
    is_complaint = bool(randint(0, 1))
    if not is_complaint:
        response_q = llm_quesitons.invoke(input=f"""Вы являетесь экспертом по кибербезопасности и специалистом из технической поддержки, отвечаете из базы знаний по кибербезопасности. Тебе задают вопросы пользователи, подготовленные и нет в отношении кибербезопасности. Тебе будет предоставлен домен в области кибербезопасности, и вам необходимо создать вопрос, относящихся к этому домену. Вопрос должен иметь отношение к домену, вопрос задает человек и хочет получить релевантный ответ от техподдержки, а ответ должен быть точным и информативным. 
        <few_shot>domain:"phishing attacks" Question: "Я нажал на незнакомую ссылку и ввел пароль от Вконтакте. Что это такое, меня обманули?"</few_shot>. 
        Теперь тебе нужно сгенерировать вопрос, относящийся к следующему домену {selected_domain}, на русском языке."""
         )
        response_a = llm_answer.invoke(input=f"Вы являетесь экспертом по кибербезопасности и специалистом из технической поддержки, отвечаете из базы знаний по кибербезопасности. Тебе задают вопросы пользователи, подготовленные и нет в отношении кибербезопасности. Тебе надо ответить на вопрос: {response_q.content}, вопрос из домена: {selected_domain}. Ответь исчерпывающе на вопрос пользователя.")
    else:
        response_q = llm_quesitons.invoke(input=f"""Вы являетесь специалистом из технической поддержки с базой знаний по кибербезопасности, отвечаете из базы знаний по кибербезопасности. Тебе задают вопросы пользователи в форме жалобы. Пользователю что-то не нравиться, его вопрос не содержит конкретных намерений что-либо уточнить или вопрос не относится к сфере кибербезопасности. Пользователь просто жалуется.
        Теперь вам нужно сгенерировать вопрос, относящихся к следующему домену {selected_domain}, на русском языке.""")
        
    print(response.content)
    qa_df.loc[i] = [selected_domain, response_q.content, response_a.content if not is_complaint else None,  is_complaint]
    qa_df.to_csv(Path("qa_dataset.csv"), index=False)
    # break

In [ ]:
from ds_generate import add_row

next(add_row(llm_quesitons, llm_answer, domains, qa_df, 0))

In [74]:
df = pd.read_json(Path("dataset.json"))

In [75]:
df

,ticket_id,user_text,expected,answer
0,TICKET-C001,"Третий день не могу войти в аккаунт, поддержка...",complaint_escalation,None
1,TICKET-C002,"Сервис постоянно падает, работать невозможно, ...",complaint_escalation,None
2,TICKET-C003,"С меня списали деньги дважды, а ответа от вас ...",complaint_escalation,None
3,TICKET-C004,"Каждый раз при оплате ошибка, вы вообще тестир...",complaint_escalation,None
4,TICKET-C005,"Обещали решить проблему вчера, но ничего не из...",complaint_escalation,None
5,TICKET-C006,"После обновления все стало только хуже, полови...",complaint_escalation,None
6,TICKET-C007,Почему мой тикет закрыли без решения? Это возм...,complaint_escalation,None
7,TICKET-C008,"Я устал писать в поддержку, никто не помогает.",complaint_escalation,None
8,TICKET-C009,"Ваш бот отвечает не по теме, а оператор недост...",complaint_escalation,None
9,TICKET-C010,"Сроки решения постоянно переносятся, это уже п...",complaint_escalation,None


In [1]:
%load_ext autoreload
%autoreload 2

from support_agent.knowledge_base import HybridChromaKnowledgeBase
from pathlib import Path
from support_agent.config import Settings

settings = Settings().from_env()
kb = HybridChromaKnowledgeBase.from_markdown_files([Path("dataset/kb_doc_1.md"), Path("dataset/kb_doc_2.md"), Path("dataset/kb_doc_3.md")], settings=settings)

In [4]:
postgre_kb = HybridChromaKnowledgeBase.from_postgres( settings=settings)

In [5]:
postgre_kb.documents

[Document(metadata={'id': 1, 'title': 'One-Sentence Image Matting! DiffSynth Open Sources Text-Guided Image Layer Separation Model', 'answer': 'Mocked content'}, page_content='Mocked content'),
 Document(metadata={'id': 2, 'title': 'Consciousness Convergence Mathematics: A Transdisciplinary Framework for Substrate-Independent Awareness', 'answer': 'Mocked content'}, page_content='Mocked content'),
 Document(metadata={'id': 3, 'title': 'Introducing OptiMind, a research model designed for optimization', 'answer': 'Mocked content'}, page_content='Mocked content'),
 Document(metadata={'id': 4, 'title': 'Tracing the thoughts of a large language model', 'answer': 'Mocked content'}, page_content='Mocked content'),
 Document(metadata={'id': 5, 'title': 'Alignment Research', 'answer': 'Mocked content'}, page_content='Mocked content'),
 Document(metadata={'id': 6, 'title': 'Anthropic Economic Index report: Economic primitives', 'answer': 'Mocked content'}, page_content='Mocked content'),
 Docume

In [28]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from langchain_core.documents import Document


url = URL.create(
            "postgresql+psycopg",
            username="postgres",
            password="postgres",
            host="localhost",
            port=5432,
            database="postgres",
        )
query = "SELECT id, title, content FROM extracted_data"

engine = create_engine(url)
try:
    df = pd.read_sql_query(query, engine)
finally:
    engine.dispose()

documents: list[Document] = []
for _, row in df.iterrows():
    documents.append(
        Document(
            page_content=str(row["content"]) if "content" in row else "Mocked content",
            metadata={
                "id": row["id"],
                "title": row["title"],
                "answer": str(row["content"]) if "content" in row else "Mocked content",
            },
        )
    )



In [29]:
kb.documents


[Document(metadata={'id': 'TICKET-Q001', 'answer': 'Нажмите «Забыли пароль» на странице входа, введите email и перейдите по ссылке из письма для установки нового пароля.', 'source': 'dataset/kb_doc_1.md'}, page_content='Answer: Нажмите «Забыли пароль» на странице входа, введите email и перейдите по ссылке из письма для установки нового пароля.'),
 Document(metadata={'id': 'TICKET-Q002', 'answer': 'Откройте «Настройки -> Оплата», добавьте новую карту и выберите ее как основной способ оплаты.', 'source': 'dataset/kb_doc_1.md'}, page_content='Answer: Откройте «Настройки -> Оплата», добавьте новую карту и выберите ее как основной способ оплаты.'),
 Document(metadata={'id': 'TICKET-Q003', 'answer': 'Перейдите в «Финансы -> Документы», выберите нужный месяц и нажмите «Скачать акт» или «Счет-фактуру».', 'source': 'dataset/kb_doc_1.md'}, page_content='Answer: Перейдите в «Финансы -> Документы», выберите нужный месяц и нажмите «Скачать акт» или «Счет-фактуру».'),
 Document(metadata={'id': 'TICK

In [15]:
kb.search("как переключить тему в приложении?")

[{'id': 'TICKET-Q024',
  'title': '',
  'content': 'Перейдите в «Профиль -> Оформление» и выберите режим «Темная тема».',
  'source': 'dataset/kb_doc_3.md'},
 {'id': 'TICKET-Q002',
  'title': '',
  'content': 'Откройте «Настройки -> Оплата», добавьте новую карту и выберите ее как основной способ оплаты.',
  'source': 'dataset/kb_doc_1.md'},
 {'id': 'TICKET-Q015',
  'title': '',
  'content': 'Перейдите в «Разработчикам -> API-ключи», нажмите «Создать ключ» и сохраните его в безопасном месте.',
  'source': 'dataset/kb_doc_2.md'}]

In [18]:
from support_agent.graph import app



ImportError: cannot import name 'app' from 'support_agent.graph' (/Users/zheka/Documents/ML/agents/ccz_test/support_agent/graph.py)